In [ ]:
# ===== Pacotes / Packages =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import itertools

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import gaussian_kde
from sklearn.pipeline import make_pipeline

In [ ]:
# ===== 1. Ler os dados .XLSX / Read .XLSX data =====
dados = pd.read_excel("NAME.xlsx")
print(dados.head())

In [ ]:
# === Ler os dados .CSV / Read .CSV data ===
dados = pd.read_csv('NAME.csv', sep=',')  # especifica o separador / specifies the separator
print(dados.head())

Se amostras independentes (não agrupadas) / For independent (ungrouped) samples.

In [ ]:
# A classe deve estar na primeira coluna / The class must be in the first column
classe = dados.iloc[:, 0].astype(str)  # forçar string/categórica / force string/categorical
X = dados.iloc[:, 1:]  # preditoras / predictors

# ===== 2. Garantir que preditoras sejam numéricas / Ensure predictors are numeric =====
X = X.apply(pd.to_numeric, errors='coerce')

# Remover colunas constantes / Remove constant columns
X = X.loc[:, X.nunique() > 1]

# Remover colunas com NA / Remove columns with NA
X = X.dropna(axis=1)

# Recombinar com a classe / Recombine with the class
dados = X.copy()
dados["Classe"] = classe

# ===== Divisão treino/teste / Training/testing division =====
X_train, X_test, y_train, y_test = train_test_split(
    X, classe, test_size=0.3, random_state=1234, stratify=classe
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
# ===== Rodar LDA / Run LDA =====
lda = LDA()
lda.fit(X_train, y_train)

In [ ]:
# ===== Stratified 5-Fold Cross-Validation =====

# Usando exatamente o mesmo modelo LDA (lda_model) definido anteriormente
model_to_evaluate = lda

# Objeto de validação cruzada estratificada
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1234
)

# Cross-validation apenas no conjunto de treino utilizando o lda_model
scores = cross_val_score(
    model_to_evaluate,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy"
)

print("===== Resultados do Stratified 5-Fold CV =====")
print(f"Acurácias por dobra (Fold): {scores}")
print(f"Acurácia média: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

Se amostras dependentes (agrupadas) / For dependent (grouped) samples.

In [ ]:
# A primeira coluna é o ID (Grupo), a segunda é a Classe # The first column is the ID (Group), the second is the Class
grupos = dados.iloc[:, 0].astype(str)
classe = dados.iloc[:, 1].astype(str)  # forçar string/categórica / force string/categorical
X = dados.iloc[:, 2:]  # preditoras / predictors (a partir da terceira coluna)

# ===== 2. Garantir que preditoras sejam numéricas / Ensure predictors are numeric =====
X = X.apply(pd.to_numeric, errors='coerce')

# Remover colunas constantes / Remove constant columns
X = X.loc[:, X.nunique() > 1]

# Remover colunas com NA / Remove columns with NA
X = X.dropna(axis=1)

# Recombinar os dados limpos para consistência se necessário
dados_limpos = X.copy()
dados_limpos["Grupo"] = grupos
dados_limpos["Classe"] = classe

# ===== Divisão treino/teste estratificada por grupo / Stratified Group Split =====
# Usamos StratifiedGroupKFold com n_splits=3 (ou similar) para obter um split de treino/teste independente por grupo
sgkf_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1234)
train_idx, test_idx = next(sgkf_split.split(X, classe, groups=grupos))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = classe.iloc[train_idx], classe.iloc[test_idx]
grupos_train, grupos_test = grupos.iloc[train_idx], grupos.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(f"Grupos no Treino: {grupos_train.nunique()} | Grupos no Teste: {grupos_test.nunique()}")
# Garante interseção zero entre grupos de treino e teste
print("Interseção de grupos entre treino e teste:", set(grupos_train).intersection(set(grupos_test)))

 # A interseção de grupos entre treino e teste é estritamente vazia (set()), o que garante que nenhuma amostra do mesmo grupo foi misturada entre treino e teste.
 # The intersection of groups between the training and test sets is strictly empty (set()), which ensures that no samples from the same group have been mixed between the training and test sets.

In [ ]:
# ===== Rodar LDA / Run LDA =====
lda = LDA()
lda.fit(X_train, y_train)

In [ ]:
# ===== Stratified Group 5-Fold Cross-Validation =====

# Usando exatamente o mesmo modelo LDA definido acima
model_to_evaluate = lda

# Objeto de validação cruzada que respeita os grupos (evita misturar o mesmo ID no treino e teste)
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=1234
)

# Cross-validation apenas no conjunto de treino respeitando os grupos
scores = cross_val_score(
    model_to_evaluate,
    X_train,
    y_train,
    groups=grupos_train,
    cv=sgkf,
    scoring="accuracy"
)

print("===== Resultados do Stratified Group 5-Fold CV =====")
print(f"Acurácias por dobra (Fold): {scores}")
print(f"Acurácia média: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

Continue

In [ ]:
# ===== Predição / Prediction =====
pred_train = lda.predict(X_train)
pred_test = lda.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, pred_train))
print("Test Accuracy:", accuracy_score(y_test, pred_test))


In [ ]:
X_lda = lda.transform(X)
num_ld = X_lda.shape[1]

# Calculate explained variance ratio
explained_variance = lda.explained_variance_ratio_ * 100

# === Biplot (LD1 vs LD2) ===
# Ensure there are at least two LD components to plot LD1 vs LD2
if num_ld >= 2:
    scores = X_lda
    plt.figure(figsize=(7,6))

    # cor dos pontos pela classe / color of points by class
    sns.scatterplot(x=scores[:,0], y=scores[:,1], hue=classe, palette="tab10", alpha=0.7)

    # Calculate loading magnitudes for LD1 and LD2
    # lda.scalings_ contains the coefficients of the linear discriminants, analogous to PCA loadings
    loading_magnitudes = np.sqrt(lda.scalings_[:,0]**2 + lda.scalings_[:,1]**2)

    # Get indices of top 5 loadings
    top_5_indices = np.argsort(loading_magnitudes)[::-1][:5]

    for i in top_5_indices:
        var = X.columns[i]
        # Scale the arrows for better visualization if needed
        plt.arrow(0, 0, lda.scalings_[i,0]*max(scores[:,0])*0.5, lda.scalings_[i,1]*max(scores[:,1])*0.5,
                  color="red", alpha=0.7, head_width=0.1)
        plt.text(lda.scalings_[i,0]*max(scores[:,0])*0.5*1.1,
                 lda.scalings_[i,1]*max(scores[:,1])*0.5*1.1,
                 var, color="red", ha="center", va="center")

    plt.title("Biplot LDA (Top 5 Loadings)")
    plt.xlabel(f"LD1 ({explained_variance[0]:.2f}%) Kishan")
    plt.ylabel(f"LD2 ({explained_variance[1]:.2f}%) Kishan")
    plt.legend(title="Class")
    plt.grid(True)
    plt.show()
else:
    print("Not enough LD components to create a Biplot (LD1 vs LD2). At least 2 components are required.")

In [ ]:
# ===== 6. Gráfico de discriminação / Discrimination Chart =====

X_lda = lda.transform(X)
num_ld = X_lda.shape[1]

# Calculate explained variance ratio
explained_variance = lda.explained_variance_ratio_ * 100

# Loop through all unique combinations of LD components
for i in range(num_ld):
    for j in range(i + 1, num_ld):
        ld_col_i = f"LD{i+1}"
        ld_col_j = f"LD{j+1}"

        lda_df = pd.DataFrame({
            ld_col_i: X_lda[:, i],
            ld_col_j: X_lda[:, j],
            "Class": classe
        })

        plt.figure(figsize=(8,6))
        sns.scatterplot(data=lda_df, x=ld_col_i, y=ld_col_j, hue="Class", alpha=0.7, s=60)
        plt.title(f"Discrimination via LDA ({ld_col_i} vs {ld_col_j})")
        plt.xlabel(f"{ld_col_i} ({explained_variance[i]:.2f}%)")
        plt.ylabel(f"{ld_col_j} ({explained_variance[j]:.2f}%)")
        plt.legend(title="Class", bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.show()


if num_ld == 1:
    lda_df = pd.DataFrame({
        "LD1": X_lda[:, 0],
        "Class": classe
    })

    plt.figure(figsize=(8,6))
    # Turn off default legend from kdeplot
    ax = sns.kdeplot(data=lda_df, x="LD1", hue="Class", fill=True, alpha=0.5, legend=False)
    plt.title(f"Discrimination via LDA Evaluating Growth")
    plt.xlabel(f"LD1 ({explained_variance[0]:.2f}%)")
    plt.ylabel("Density")

    # Manually create legend for kdeplot
    unique_classes = classe.unique()
    palette = sns.color_palette() # Get default color palette
    handles = [mpatches.Patch(color=palette[i], label=cls) for i, cls in enumerate(unique_classes)]
    ax.legend(handles=handles, title="Class", bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()


In [ ]:
#Caso seja 2 classes = salvar curvas

# 1. Recriar o dataframe base de LD1 e classes
lda_df = pd.DataFrame({
    "LD1": X_lda[:, 0],
    "Class": classe
})

# 2. Calcular o KDE expandindo o limite de X para a densidade ir a zero
kde_data_list = []

for cls in lda_df["Class"].unique():
    subset = lda_df[lda_df["Class"] == cls]["LD1"]
    if len(subset) > 1:
        kde = gaussian_kde(subset)

        # Adiciona uma margem de 3 vezes o desvio padrão (ou um valor fixo)
        # para garantir que a estimativa da curva chegue a zero nas pontas
        margem = 5.0
        x_min = subset.min() - margem
        x_max = subset.max() + margem

        x_vals = np.linspace(x_min, x_max, 300)
        y_vals = kde(x_vals)

        df_kde_cls = pd.DataFrame({
            'Classe': cls,
            'LD1_Eixo_X': x_vals,
            'Densidade_Eixo_Y': y_vals
        })
        kde_data_list.append(df_kde_cls)

# Concatenar todos os novos dados com as pontas suavizadas até zero
df_densidades_completas = pd.concat(kde_data_list, ignore_index=True) if kde_data_list else pd.DataFrame()

# 3. Salvar o arquivo excel com as abas separadas por classe
nome_arquivo_classes = "dados_ld1_densidade_por_classe.xlsx"
classes_unicas = df_densidades_completas["Classe"].unique()

with pd.ExcelWriter(nome_arquivo_classes, engine='openpyxl') as writer:
    for cls in classes_unicas:
        df_classe = df_densidades_completas[df_densidades_completas["Classe"] == cls][["LD1_Eixo_X", "Densidade_Eixo_Y"]]
        df_classe.columns = ["LD1", "Densidade"]
        nome_aba = str(cls)[:31]
        df_classe.to_excel(writer, sheet_name=nome_aba, index=False)

print(f"Arquivo '{nome_arquivo_classes}' gerado com sucesso com as curvas corrigidas descendo até zero!")

In [ ]:
# ===== 7. Loadings =====
loadings = lda.scalings_
num_ld = loadings.shape[1]

# Loop through all unique combinations of LD components
if num_ld >= 2:
    for i in range(num_ld):
        for j in range(i + 1, num_ld):
            ld_col_i = f"LD{i+1}"
            ld_col_j = f"LD{j+1}"

            loadings_df = pd.DataFrame(loadings[:, [i, j]],
                                       columns=[ld_col_i, ld_col_j],
                                       index=X.columns).reset_index()
            loadings_df.rename(columns={"index": "Variavel"}, inplace=True)

            # Calculate the magnitude of the loading vector for each variable
            loadings_df['Magnitude'] = np.sqrt(loadings_df[ld_col_i]**2 + loadings_df[ld_col_j]**2)

            # Select the top 10 variables by magnitude
            top_10_loadings = loadings_df.sort_values(by='Magnitude', ascending=False).head(10)

            plt.figure(figsize=(10, 8))
            for _, row in top_10_loadings.iterrows(): # Use top_10_loadings here
                plt.arrow(0, 0, row[ld_col_i], row[ld_col_j],
                          color="blue", alpha=0.7, head_width=0.02, length_includes_head=True)
                plt.text(row[ld_col_i] * 1.05, row[ld_col_j] * 1.05, row["Variavel"],
                         color="red", ha="center", va="center", fontsize=9)

            plt.axhline(0, color="gray", linestyle="--")
            plt.axvline(0, color="gray", linestyle="--")
            plt.title(f"Top 10 Feature loadings ({ld_col_i} vs {ld_col_j})")
            plt.xlabel(ld_col_i)
            plt.ylabel(ld_col_j)
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.show()
else:
    print("Not enough LD components to plot pairwise feature loadings. \nAt least 2 components are required for 'LDn vs LDn' plots.")


In [ ]:
# Ensure num_ld, X_lda, explained_variance, and classe are available
# (These variables are already defined in previous cells and kernel state)

if num_ld >= 3:
    print("Generating 3D discrimination plots...")

    # Get unique classes and assign a color map
    unique_classes = np.unique(classe)
    # Updated to use the recommended colormaps.get_cmap method
    colors = plt.colormaps.get_cmap('tab10')
    class_color_map = {cls: colors(i) for i, cls in enumerate(unique_classes)}

    # Generate all unique combinations of 3 distinct LD components
    ld_combinations = list(itertools.combinations(range(num_ld), 3))

    for combo in ld_combinations:
        idx1, idx2, idx3 = combo
        ld_col_name1 = f"LD{idx1+1}"
        ld_col_name2 = f"LD{idx2+1}"
        ld_col_name3 = f"LD{idx3+1}"

        # Create a temporary DataFrame for plotting
        plot_df = pd.DataFrame({
            ld_col_name1: X_lda[:, idx1],
            ld_col_name2: X_lda[:, idx2],
            ld_col_name3: X_lda[:, idx3],
            "Class": classe
        })

        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        for cls in unique_classes:
            subset = plot_df[plot_df['Class'] == cls]
            ax.scatter(
                subset[ld_col_name1],
                subset[ld_col_name2],
                subset[ld_col_name3],
                label=cls,
                color=class_color_map[cls],
                alpha=0.7,
                s=60
            )

        ax.set_title(f"Discrimination via LDA ({ld_col_name1} vs {ld_col_name2} vs {ld_col_name3})")
        ax.set_xlabel(f"{ld_col_name1} ({explained_variance[idx1]:.2f}%)")
        ax.set_ylabel(f"{ld_col_name2} ({explained_variance[idx2]:.2f}%)")
        ax.set_zlabel(f"{ld_col_name3} ({explained_variance[idx3]:.2f}%)")
        ax.legend(title="Class", bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough LD components to plot 3D graphs. At least 3 components are required.")


In [ ]:
csv_filename = 'all_loadings.csv'

# Create a DataFrame from the full loadings array
# 'loadings' is already defined from `lda.scalings_`
# The number of columns in 'loadings' corresponds to num_ld
ld_column_names = [f'LD{i+1}' for i in range(loadings.shape[1])]
full_loadings_df = pd.DataFrame(loadings, columns=ld_column_names, index=X.columns)

# Save the full loadings DataFrame to CSV
full_loadings_df.to_csv(csv_filename, index=True) # index=True to save variable names as a column
print(f"All loadings saved to {csv_filename}")

In [ ]:
# Ensure loadings and X.columns are available from previous executions

# Get the number of LD components
num_ld = loadings.shape[1]

# Loop through each LD component to find and save top 20 loadings
for k in range(num_ld):
    ld_col_name = f"LD{k+1}"

    # Create a DataFrame for the current LD component's loadings
    loadings_df_current_ld = pd.DataFrame({
        ld_col_name: loadings[:, k],
        "Variavel": X.columns
    })

    # Calculate absolute LD values and sort to find the top loadings
    loadings_df_current_ld['Abs_LD'] = loadings_df_current_ld[ld_col_name].abs()

    # Select the top 20 loadings for the current LD component
    top_20_loadings_current = loadings_df_current_ld.sort_values(by='Abs_LD', ascending=False).head(20)

    # Exclude the temporary 'Abs_LD' column
    top_20_loadings_current = top_20_loadings_current.drop(columns=['Abs_LD'])

    # Sort in descending order according to the loading value
    top_20_loadings_current = top_20_loadings_current.sort_values(by=ld_col_name, ascending=False)

    # Display the top 20 loadings table for the current LD component
    print(f"\nTop 20 Loadings for {ld_col_name}:")
    print(top_20_loadings_current[['Variavel', ld_col_name]])

    # Save the top 20 loadings to a CSV file
    csv_filename = f'top_20_loadings_{ld_col_name}.csv'
    top_20_loadings_current.to_csv(csv_filename, index=False)
    print(f"Top 20 loadings for {ld_col_name} saved to {csv_filename}")

In [ ]:
# ---- Matriz de confusão para o conjunto de teste / Confusion matrix for the test set ----
cm = confusion_matrix(y_test, pred_test, labels=np.unique(classe))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(classe))
disp.plot(cmap="Blues", values_format="d")
plt.xticks(rotation=90) # Rotate x-axis labels
plt.title("Confusion Matrix - Test")
plt.show()

print("\nClassification Report - Test:\n")
print(classification_report(y_test, pred_test, target_names=np.unique(classe)))

In [ ]:
# Loop through all LD components
for k in range(num_ld):
    ld_col_name = f"LD{k+1}"

    # Re-create loadings_df for the current LD case
    loadings_df = pd.DataFrame({
        ld_col_name: loadings[:, k],
        "Variavel": X.columns
    })

    # Convert 'Variavel' (wavenumber) column to numeric
    loadings_df['Variavel_numeric'] = pd.to_numeric(loadings_df['Variavel'])

    # Sort by wavenumber
    loadings_df = loadings_df.sort_values('Variavel_numeric')

    # Selecionar os 10 maiores loadings (valor absoluto) para o LD atual
    top10 = loadings_df.reindex(
        loadings_df[ld_col_name].abs().sort_values(ascending=False).index
    ).head(10)

    plt.figure(figsize=(12, 6))

    # Barras
    plt.bar(loadings_df['Variavel_numeric'],
            loadings_df[ld_col_name],
            width=1.0,
            color='steelblue')

    # Pontos vermelhos
    plt.scatter(top10['Variavel_numeric'],
                top10[ld_col_name],
                color='red',
                s=10,
                zorder=5,
                label=f'Top 10 |{ld_col_name}|')

    # Adicionar rótulos com número de onda
    for _, row in top10.iterrows():
        x = row['Variavel_numeric']
        y = row[ld_col_name]

        # deslocamento vertical automático
        offset = 0.02 * loadings_df[ld_col_name].max()
        if y < 0:
            offset = -offset

        plt.text(x,
                 y + offset,
                 f"{int(x)}",
                 ha='center',
                 va='bottom' if y > 0 else 'top',
                 fontsize=9,
                 color='darkred')

    #plt.title(f'Loadings ({ld_col_name}) vs. Raman Shift (cm$^{{-1}}$)')
    plt.xlabel('Raman Shift (cm$^{{-1}}$)')
    plt.ylabel(f'{ld_col_name} Loading')
    #plt.grid(True, linestyle='--', alpha=0.7)
    #plt.legend()

    plt.show()

Bagged LDA (Bootstrap)

Se amostras independentes (não agrupadas) / For independent (ungrouped) samples.

In [ ]:
# ===== Bootstrap Aggregation (Bagging) com o modelo LDA anterior =====
from sklearn.base import clone

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(classe)

n_bootstrap = 1000
n_samples = len(X)

rng = np.random.default_rng(42)

# Store the trained LDA models
bagged_models = []

# Store OOB predictions for each original sample
oob_predictions = [[] for _ in range(n_samples)]

for bootstrap in range(n_bootstrap):

    # Generate bootstrap sample
    train_idx = rng.choice(
        n_samples,
        size=n_samples,
        replace=True
    )

    # Identify Out-of-Bag samples
    bootstrap_mask = np.zeros(n_samples, dtype=bool)
    bootstrap_mask[train_idx] = True

    oob_idx = np.where(~bootstrap_mask)[0]

    if len(oob_idx) == 0:
        continue

    # Training data
    X_train = X.iloc[train_idx]
    y_train = y[train_idx]

    # Clona exatamente a configuração do modelo LDA definido anteriormente
    lda_boot = clone(lda)
    lda_boot.fit(X_train, y_train)

    # Store the trained model
    bagged_models.append(lda_boot)

    # OOB prediction
    X_oob = X.iloc[oob_idx]
    y_pred_oob = lda_boot.predict(X_oob)

    # Store predictions for each OOB sample
    for sample_idx, prediction in zip(oob_idx, y_pred_oob):
        oob_predictions[sample_idx].append(prediction)


print("\n===== Bootstrap Aggregation Results =====")
print(f"Requested bootstrap models: {n_bootstrap}")
print(f"Trained LDA models: {len(bagged_models)}")
print(f"Original samples: {n_samples}")

oob_counts = [len(predictions) for predictions in oob_predictions]

print(f"Minimum OOB predictions per sample: {np.min(oob_counts)}")
print(f"Maximum OOB predictions per sample: {np.max(oob_counts)}")
print(f"Mean OOB predictions per sample: {np.mean(oob_counts):.2f}")

In [ ]:
# ===== OOB Ensemble Prediction =====

oob_final_predictions = []
oob_true_labels = []

for sample_idx in range(n_samples):

    predictions = oob_predictions[sample_idx]

    if len(predictions) > 0:

        # Majority vote
        values, counts = np.unique(predictions, return_counts=True)
        final_prediction = values[np.argmax(counts)]

        oob_final_predictions.append(final_prediction)
        oob_true_labels.append(y[sample_idx])


oob_true_labels = np.array(oob_true_labels)
oob_final_predictions = np.array(oob_final_predictions)


# ===== OOB Ensemble Performance =====

oob_accuracy = accuracy_score(
    oob_true_labels,
    oob_final_predictions
)

print("\n===== OOB Ensemble Performance =====")
print(f"OOB Ensemble Accuracy: {oob_accuracy:.4f}")


print("\nClassification Report - OOB Ensemble:\n")

print(
    classification_report(
        oob_true_labels,
        oob_final_predictions,
        target_names=label_encoder.inverse_transform(np.unique(y))
    )
)


# ===== Confusion Matrix =====

cm = confusion_matrix(
    oob_true_labels,
    oob_final_predictions,
    labels=np.unique(y)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.inverse_transform(np.unique(y))
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.xticks(rotation=90)
plt.title("Confusion Matrix - OOB Bagged LDA")
plt.show()

Se amostras dependentes (agrupadas) / For dependent (grouped) samples.

If the number of IDs (group) is very small, group-level bootstrapping may be unstable: few unique groups reduce the diversity among bootstrap samples and may limit the benefit of bagging. In this case, a simpler and more robust alternative is to use only StratifiedGroupKFold with a single LDA model per fold, without aggregation by voting.

In [ ]:
# ===== Bagged LDA com Estratificação e Respeito aos Grupos (Paciente) =====
from sklearn.base import clone
import pandas as pd

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(classe)

# Definir grupos de forma explícita conforme dados limpos ou de agrupamento anteriores
groups = grupos.to_numpy()

n_bootstrap = 1000
n_samples = len(X)

rng = np.random.default_rng(42)

# Guardar os modelos LDA treinados
bagged_models = []

# Guardar as predições Out-of-Bag (OOB) para cada amostra original
oob_predictions = [[] for _ in range(n_samples)]

# Criar mapeamento de observações pertencentes a cada grupo para amostragem agrupada
group_info = pd.DataFrame({
    'Group': groups,
    'Class': y
}).drop_duplicates()

unique_groups = group_info['Group'].to_numpy()
unique_group_classes = group_info['Class'].to_numpy()

for bootstrap in range(n_bootstrap):
    # Amostragem por Bootstrapping ao nível do grupo (estratificada por classe do grupo)
    sampled_groups = []
    for current_class in np.unique(y):
        class_groups = group_info.loc[group_info['Class'] == current_class, 'Group'].to_numpy()
        sampled_class_groups = rng.choice(
            class_groups,
            size=len(class_groups),
            replace=True
        )
        sampled_groups.extend(sampled_class_groups)

    sampled_groups = np.array(sampled_groups)

    # Encontrar os índices originais das amostras correspondentes aos grupos sorteados
    train_idx = []
    for group in sampled_groups:
        group_idx = np.where(groups == group)[0]
        train_idx.extend(group_idx)
    train_idx = np.array(train_idx, dtype=int)

    # Identificar quais grupos ficaram de fora (Out-of-Bag ao nível do grupo)
    oob_groups = np.setdiff1d(unique_groups, sampled_groups)
    oob_idx = np.where(np.isin(groups, oob_groups))[0]

    if len(oob_idx) == 0:
        continue

    # Dados de treino baseados no bootstrap agrupado
    X_train_boot = X.iloc[train_idx]
    y_train_boot = y[train_idx]

    # Clona exatamente a configuração do modelo LDA anterior
    lda_boot = clone(lda)
    lda_boot.fit(X_train_boot, y_train_boot)

    # Armazenar modelo treinado
    bagged_models.append(lda_boot)

    # Predição nos dados Out-Of-Bag (OOB)
    X_oob = X.iloc[oob_idx]
    y_pred_oob = lda_boot.predict(X_oob)

    # Armazenar predições OOB
    for idx, prediction in zip(oob_idx, y_pred_oob):
        oob_predictions[idx].append(prediction)

print("\n===== Bootstrap Aggregation Results (Group-Aware) =====")
print(f"Requested bootstrap models: {n_bootstrap}")
print(f"Trained LDA models: {len(bagged_models)}")
print(f"Original samples: {n_samples}")

oob_counts = [len(predictions) for predictions in oob_predictions]

print(f"Minimum OOB predictions per sample: {np.min(oob_counts)}")
print(f"Maximum OOB predictions per sample: {np.max(oob_counts)}")
print(f"Mean OOB predictions per sample: {np.mean(oob_counts):.2f}")

In [ ]:
# ===== OOB Ensemble Prediction =====

oob_final_predictions = []
oob_true_labels = []

for sample_idx in range(n_samples):

    predictions = oob_predictions[sample_idx]

    if len(predictions) > 0:

        # Majority vote
        values, counts = np.unique(predictions, return_counts=True)
        final_prediction = values[np.argmax(counts)]

        oob_final_predictions.append(final_prediction)
        oob_true_labels.append(y[sample_idx])


oob_true_labels = np.array(oob_true_labels)
oob_final_predictions = np.array(oob_final_predictions)


# ===== OOB Ensemble Performance =====

oob_accuracy = accuracy_score(
    oob_true_labels,
    oob_final_predictions
)

print("\n===== OOB Ensemble Performance =====")
print(f"OOB Ensemble Accuracy: {oob_accuracy:.4f}")


print("\nClassification Report - OOB Ensemble:\n")

print(
    classification_report(
        oob_true_labels,
        oob_final_predictions,
        target_names=label_encoder.inverse_transform(np.unique(y))
    )
)


# ===== Confusion Matrix =====

cm = confusion_matrix(
    oob_true_labels,
    oob_final_predictions,
    labels=np.unique(y)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.inverse_transform(np.unique(y))
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.xticks(rotation=90)
plt.title("Confusion Matrix - OOB Bagged LDA")
plt.show()